<a href="https://colab.research.google.com/github/O7car/Fisica/blob/main/Anyones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install numpy matplotlib scipy ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [4]:
def pep_violation(A=1.6, sigma=0.6, V=0.9, t0=2.0, tmax=5.0):
    def lambda_t(t):
        return A * np.exp(-(t - t0)**2 / (2 * sigma**2))
    def rhs(t, y):
        cA = y[0] + 1j*y[1]
        cS = y[2] + 1j*y[3]
        lam = lambda_t(t)
        dA = -1j * lam * V * cS
        dS = -1j * lam * V * cA
        return [dA.real, dA.imag, dS.real, dS.imag]
    y0 = [1.0, 0.0, 0.0, 0.0]
    t_eval = np.linspace(0, tmax, 500)
    sol = solve_ivp(rhs, (0, tmax), y0, t_eval=t_eval, method='RK45', rtol=1e-8)
    t = sol.t
    cS = sol.y[2] + 1j*sol.y[3]
    prob_S = np.abs(cS)**2
    return t, prob_S

def plot_pep(A, sigma, V):
    t, prob = pep_violation(A, sigma, V)
    plt.figure(figsize=(8,4))
    plt.plot(t, prob, 'b-', linewidth=2)
    plt.xlabel('Tiempo (u.a.)')
    plt.ylabel('Probabilidad estado simétrico')
    plt.title(f'Violación del PEP (A={A:.2f}, σ={sigma:.2f}, V={V:.2f})')
    plt.grid(True)
    plt.ylim(0, 1)
    plt.show()

interact(plot_pep,
         A=FloatSlider(min=0, max=2.5, step=0.05, value=1.6, description='Amplitud A'),
         sigma=FloatSlider(min=0.2, max=1.2, step=0.02, value=0.6, description='Anchura σ'),
         V=FloatSlider(min=0, max=1.2, step=0.02, value=0.9, description='Acoplamiento V'));

interactive(children=(FloatSlider(value=1.6, description='Amplitud A', max=2.5, step=0.05), FloatSlider(value=…

In [5]:
def cooper_to_anyon(Vsp=0.85, Ap=1.4, delta=0.0, t0=1.8, sigma=0.6, tmax=4.0):
    def lambda_t(t):
        return Ap * np.exp(-(t - t0)**2 / (2 * sigma**2))
    def rhs(t, y):
        cS = y[0] + 1j*y[1]
        cP = y[2] + 1j*y[3]
        lam = lambda_t(t)
        dS = -1j * lam * Vsp * cP
        dP = -1j * (lam * Vsp * cS + delta * cP)
        return [dS.real, dS.imag, dP.real, dP.imag]
    y0 = [1.0, 0.0, 0.0, 0.0]
    t_eval = np.linspace(0, tmax, 500)
    sol = solve_ivp(rhs, (0, tmax), y0, t_eval=t_eval, method='RK45', rtol=1e-8)
    t = sol.t
    cS = sol.y[0] + 1j*sol.y[1]
    cP = sol.y[2] + 1j*sol.y[3]
    prob_p = np.abs(cP)**2
    pS = np.abs(cS[-1])**2
    pP = np.abs(cP[-1])**2
    fase_eff = pS * 1.0 + pP * np.exp(1j * np.pi/8)
    arg = np.angle(fase_eff) / np.pi
    return t, prob_p, cS[-1], cP[-1], arg

def plot_anyon(Vsp, Ap):
    t, prob_p, cs, cp, arg = cooper_to_anyon(Vsp, Ap)
    plt.figure(figsize=(8,4))
    plt.plot(t, prob_p, 'r-', linewidth=2)
    plt.xlabel('Tiempo (u.a.)')
    plt.ylabel('Probabilidad canal p+ip (anyónico)')
    plt.title(f'Transición a anyón σ (Vsp={Vsp:.2f}, Ap={Ap:.2f})')
    plt.grid(True)
    plt.ylim(0, 1)
    plt.show()
    print(f"Coeficiente s: {cs.real:.3f} + {cs.imag:.3f}j  (prob = {np.abs(cs)**2:.3f})")
    print(f"Coeficiente p: {cp.real:.3f} + {cp.imag:.3f}j  (prob = {np.abs(cp)**2:.3f})")
    print(f"Fase efectiva: {arg:.4f} π   (teórico: 0.125π)")

interact(plot_anyon,
         Vsp=FloatSlider(min=0.2, max=1.2, step=0.02, value=0.7231, description='Acoplamiento V_sp'),
         Ap=FloatSlider(min=1.0, max=2.0, step=0.02, value=1.4462, description='Amplitud A_p'));

interactive(children=(FloatSlider(value=0.7231, description='Acoplamiento V_sp', max=1.2, min=0.2, step=0.02),…

In [6]:
L = 1.0
N = 80
x = np.linspace(0, L, N)
X1, X2 = np.meshgrid(x, x, indexing='ij')

def phi_nx_ny(xx, yy, nx, ny):
    return np.sqrt(2/L) * np.sin(nx * np.pi * xx / L) * np.sqrt(2/L) * np.sin(ny * np.pi * yy / L)

phi0 = phi_nx_ny(X1, X2, 1, 1)
phi1 = phi_nx_ny(X1, X2, 1, 2)

def density_2d(theta):
    term1 = np.exp(1j * theta) * (phi0 * phi1)
    term2 = np.exp(-1j * theta) * (phi1 * phi0)
    psi = (term1 + term2) / np.sqrt(2)
    return np.abs(psi)**2

def plot_2d(theta):
    rho = density_2d(theta)
    plt.figure(figsize=(6,5))
    plt.contourf(X1, X2, rho, levels=50, cmap='hot')
    plt.plot([0, L], [0, L], 'w--', linewidth=2, label='diagonal')
    plt.colorbar(label='Densidad')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title(f'θ = {theta/np.pi:.3f}π')
    plt.legend()
    plt.show()

interact(plot_2d,
         theta=FloatSlider(min=0, max=np.pi/2, step=np.pi/64, value=np.pi/8,
                           description='θ (rad)', continuous_update=False));

interactive(children=(FloatSlider(value=0.39269908169872414, continuous_update=False, description='θ (rad)', m…

In [7]:
B1 = np.array([[np.exp(-1j*np.pi/8), 0],
               [0, np.exp(1j*3*np.pi/8)]], dtype=complex)
B2 = (1/np.sqrt(2)) * np.array([[np.exp(1j*np.pi/8), np.exp(-1j*3*np.pi/8)],
                                [np.exp(-1j*3*np.pi/8), np.exp(1j*np.pi/8)]], dtype=complex)
comm = B1 @ B2 - B2 @ B1
print("Conmutador [B1,B2]:\n", np.round(comm, 3))
print("Norma:", np.linalg.norm(comm))
print("B1 B2 B1 - B2 B1 B2:\n", np.round(B1@B2@B1 - B2@B1@B2, 3))
print("¿Es nula?", np.allclose(B1@B2@B1, B2@B1@B2))

Conmutador [B1,B2]:
 [[ 0.   +0.j    -0.707-0.707j]
 [ 0.707+0.707j  0.   +0.j   ]]
Norma: 1.414213562373095
B1 B2 B1 - B2 B1 B2:
 [[0.+0.j 0.-0.j]
 [0.-0.j 0.+0.j]]
¿Es nula? True
